In [2]:
import os, numpy as np, pandas as pd, torch
from pathlib import Path
from PIL import Image
from concurrent.futures import ProcessPoolExecutor

ROOT = next(Path("/kaggle/input").rglob("train.csv")).parent
train_df, test_df = pd.read_csv(ROOT/"train.csv"), pd.read_csv(ROOT/"test.csv")
IMG = 224

def resolve(rel):
    """Cari path asli gambar (struktur folder Kaggle kadang bersarang: train/train/...)."""
    for p in (ROOT/rel, ROOT/Path(rel).parts[0]/rel):
        if p.exists():
            return str(p)
    raise FileNotFoundError(rel)

def load(path):
    """Baca TIFF -> grayscale -> resize IMGxIMG (uint8)."""
    try:
        return np.asarray(Image.open(path).convert("L").resize((IMG, IMG), Image.BILINEAR), dtype=np.uint8)
    except Exception:
        return np.full((IMG, IMG), 255, np.uint8)

def cache(df):
    paths = [resolve(f) for f in df.filename]
    with ProcessPoolExecutor(os.cpu_count()) as ex:
        return np.stack(list(ex.map(load, paths, chunksize=64)))

X_all = cache(train_df); y_all = train_df.label.values
X_test = cache(test_df)
print(X_all.shape, X_test.shape)

/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:982: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))


(32003, 224, 224) (7994, 224, 224)


In [3]:
import math, torchvision.transforms.v2 as T
from sklearn.model_selection import train_test_split
from transformers import BeitForImageClassification

dev = "cuda"
tr_idx, va_idx = train_test_split(np.arange(len(y_all)), test_size=0.06, stratify=y_all, random_state=42)

class DocDS(torch.utils.data.Dataset):
    """Dataset gambar dokumen grayscale ter-cache; augmentasi ringan hanya saat train."""
    def __init__(self, X, y=None, train=False):
        self.X, self.y, self.train = X, y, train
        self.aug = T.RandomAffine(degrees=2, translate=(0.02, 0.02), scale=(0.96, 1.04), fill=255)
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x = torch.from_numpy(self.X[i]).unsqueeze(0)
        if self.train: x = self.aug(x)
        x = ((x.float() / 255.0 - 0.5) / 0.5).expand(3, -1, -1)
        return (x, int(self.y[i])) if self.y is not None else x

BS = 64
mk = lambda ds, sh: torch.utils.data.DataLoader(ds, batch_size=BS, shuffle=sh, num_workers=4, pin_memory=True, drop_last=sh)
dl_tr = mk(DocDS(X_all[tr_idx], y_all[tr_idx], True), True)
dl_va = mk(DocDS(X_all[va_idx], y_all[va_idx]), False)
dl_te = mk(DocDS(X_test), False)

model = BeitForImageClassification.from_pretrained("microsoft/dit-base", num_labels=16, ignore_mismatched_sizes=True).to(dev)

def param_groups(model, lr, decay=0.7, wd=0.05):
    """Layer-wise LR decay: layer awal belajar lebih pelan, head paling cepat."""
    n, groups = model.config.num_hidden_layers, {}
    for name, p in model.named_parameters():
        if "encoder.layer." in name: lid = int(name.split("encoder.layer.")[1].split(".")[0]) + 1
        elif "embeddings" in name:   lid = 0
        else:                        lid = n + 1
        nowd = p.ndim == 1
        g = groups.setdefault((lid, nowd), {"params": [], "lr": lr * decay ** (n + 1 - lid), "weight_decay": 0.0 if nowd else wd})
        g["params"].append(p)
    return list(groups.values())

EPOCHS, LR = 10, 3e-4
opt = torch.optim.AdamW(param_groups(model, LR))
total, warm = EPOCHS * len(dl_tr), len(dl_tr)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: s / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / (total - warm))))
scaler = torch.amp.GradScaler("cuda")
crit = torch.nn.CrossEntropyLoss(label_smoothing=0.1)

config.json:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  369MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

[transformers] BeitForImageClassification LOAD REPORT from: microsoft/dit-base
Key                          | Status     | 
-----------------------------+------------+-
lm_head.weight               | UNEXPECTED | 
layernorm.bias               | UNEXPECTED | 
layernorm.weight             | UNEXPECTED | 
lm_head.bias                 | UNEXPECTED | 
beit.pooler.layernorm.bias   | MISSING    | 
classifier.weight            | MISSING    | 
beit.pooler.layernorm.weight | MISSING    | 
classifier.bias              | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors: reconstructing file:   0%|          |  0.00B /  368MB            

model.safetensors: downloading bytes:           |  0.00B            

In [4]:
@torch.no_grad()
def predict(loader):
    """Return probabilitas softmax (N,16)."""
    model.eval(); out = []
    for b in loader:
        x = b[0] if isinstance(b, (list, tuple)) else b
        with torch.autocast("cuda", dtype=torch.float16):
            out.append(model(pixel_values=x.to(dev)).logits.float().softmax(-1).cpu())
    return torch.cat(out)

os.makedirs("/kaggle/working/checkpoint", exist_ok=True)
best = 0
for ep in range(EPOCHS):
    model.train(); run = 0
    for x, y in dl_tr:
        x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = crit(model(pixel_values=x).logits, y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        run += loss.item()
    acc = (predict(dl_va).argmax(1).numpy() == y_all[va_idx]).mean()
    print(f"epoch {ep+1}/{EPOCHS} loss {run/len(dl_tr):.4f} val_acc {acc:.4f}")
    if acc > best:
        best = acc; torch.save(model.state_dict(), "/kaggle/working/checkpoint/best.pt")

/tmp/ipykernel_50/4063448351.py:22: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step()


epoch 1/10 loss 1.5070 val_acc 0.7683
epoch 2/10 loss 1.1612 val_acc 0.8100
epoch 3/10 loss 1.0539 val_acc 0.8303
epoch 4/10 loss 0.9828 val_acc 0.8563
epoch 5/10 loss 0.9105 val_acc 0.8683
epoch 6/10 loss 0.8486 val_acc 0.8631
epoch 7/10 loss 0.7848 val_acc 0.8787
epoch 8/10 loss 0.7351 val_acc 0.8792
epoch 9/10 loss 0.6988 val_acc 0.8803
epoch 10/10 loss 0.6807 val_acc 0.8813


In [ ]:
import gc
import timm

@torch.no_grad()
def predict_cnn(model, loader):
    model.eval()
    predictions = []

    for batch in loader:
        images = batch[0] if isinstance(batch, (tuple, list)) else batch
        images = images.to(dev, non_blocking=True)
        images = images.contiguous(memory_format=torch.channels_last)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(images)

        predictions.append(logits.float().softmax(dim=1).cpu())

    return torch.cat(predictions, dim=0)

gc.collect()
torch.cuda.empty_cache()

dl2_tr, dl2_va, dl2_te = dl_tr, dl_va, dl_te

cnn = timm.create_model(
    "convnext_tiny.fb_in22k_ft_in1k",
    pretrained=True,
    num_classes=16,
    drop_path_rate=0.1,
).to(dev).to(memory_format=torch.channels_last)

EP2 = 8
opt2 = torch.optim.AdamW(cnn.parameters(), lr=2e-4, weight_decay=0.05)
tot2, warm2 = EP2 * len(dl2_tr), len(dl2_tr)

sched2 = torch.optim.lr_scheduler.LambdaLR(
    opt2,
    lambda s: s / warm2 if s < warm2 else 0.5 * (
        1 + math.cos(math.pi * (s - warm2) / (tot2 - warm2))
    ),
)

scaler2 = torch.amp.GradScaler("cuda")
best2 = 0

for ep in range(EP2):
    cnn.train()
    run = 0

    for x, y in dl2_tr:
        x = x.to(dev, non_blocking=True).contiguous(
            memory_format=torch.channels_last
        )
        y = y.to(dev, non_blocking=True)

        with torch.autocast("cuda", dtype=torch.float16):
            loss = crit(cnn(x), y)

        opt2.zero_grad(set_to_none=True)
        scaler2.scale(loss).backward()
        scaler2.unscale_(opt2)
        torch.nn.utils.clip_grad_norm_(cnn.parameters(), 1.0)
        scaler2.step(opt2)
        scaler2.update()
        sched2.step()
        run += loss.item()

    acc = (
        predict_cnn(cnn, dl2_va).argmax(1).numpy() == y_all[va_idx]
    ).mean()
    print(f"epoch {ep+1}/{EP2} loss {run/len(dl2_tr):.4f} val_acc {acc:.4f}")

    if acc > best2:
        best2 = acc
        torch.save(cnn.state_dict(), "/kaggle/working/checkpoint/cnn_best.pt")

epoch 1/8 loss 1.7895 val_acc 0.7569
epoch 2/8 loss 1.1523 val_acc 0.8308
epoch 3/8 loss 0.9903 val_acc 0.8532
epoch 4/8 loss 0.8874 val_acc 0.8662
epoch 5/8 loss 0.8035 val_acc 0.8803
epoch 6/8 loss 0.7313 val_acc 0.8850
epoch 7/8 loss 0.6831 val_acc 0.8870
epoch 8/8 loss 0.6649 val_acc 0.8933


In [ ]:
cnn.load_state_dict(torch.load("/kaggle/working/checkpoint/cnn_best.pt"))
p_cnn_va = predict_cnn(cnn, dl2_va)
p_cnn_te = predict_cnn(cnn, dl2_te)

In [ ]:
yva = y_all[va_idx]

best_w, best_acc = 0, 0
for w in np.arange(0, 1.01, 0.1):
    a = ((w * p_dit_va + (1 - w) * p_cnn_va).argmax(1).numpy() == yva).mean()
    print(f"w_dit={w:.1f} val_acc={a:.4f}")
    if a > best_acc: best_w, best_acc = w, a

pred = (best_w * p_dit_te + (1 - best_w) * p_cnn_te).argmax(1).numpy()
pd.DataFrame({"id": test_df.id, "label": pred}).to_csv("/kaggle/working/bismillah.csv", index=False)
print("best weight", best_w, "val_acc", best_acc)

w_dit=0.0 val_acc=0.8933
w_dit=0.1 val_acc=0.8948
w_dit=0.2 val_acc=0.8948
w_dit=0.3 val_acc=0.8980
w_dit=0.4 val_acc=0.8969
w_dit=0.5 val_acc=0.8943
w_dit=0.6 val_acc=0.8938
w_dit=0.7 val_acc=0.8886
w_dit=0.8 val_acc=0.8860
w_dit=0.9 val_acc=0.8824
w_dit=1.0 val_acc=0.8813
best weight 0.30000000000000004 val_acc 0.8979698073919833
